In [3]:
import os, gc, re, json, psutil, warnings, random
from pathlib import Path
import numpy as np
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from mne.io import read_raw_edf

warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
SHHS1_EDF = Path(r"C:\Users\Zubair\Desktop\VIT\shhs\polysomnography\edfs\shhs1")
SHHS1_XML = Path(r"C:\Users\Zubair\Desktop\VIT\shhs\polysomnography\annotations-events-nsrr\shhs1")
SHHS2_EDF = Path(r"C:\Users\Zubair\Desktop\VIT\shhs\polysomnography\edfs\shhs2")
SHHS2_XML = Path(r"C:\Users\Zubair\Desktop\VIT\shhs\polysomnography\annotations-events-nsrr\shhs2")

OUT_DIR    = Path("./npz_dataset_4class_shhs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FS      = 100.0
WINDOW_SEC     = 60.0
STRIDE_SEC     = 30.0
MAX_MEMORY_PCT = 85
VAL_RATIO      = 0.20
SEED           = 42

# Labels: 0=ObA, 1=CnA, 2=Hyp, 3=None
LBL_MAP = {"obstructive_apnea":0, "central_apnea":1, "hypopnea":2, "none":3}

# Channel aliases to resolve SHHS variations
CANDIDATES = {
    "NEW AIR":  [r'NEW\s*AIR', r'AIRFLOW', r'NASAL\s*(PRES|FLOW|AIR)', r'NAF'],
    "THOR RES": [r'THOR(\s*RES)?', r'CHEST', r'THORAX'],
    "ABDO RES": [r'ABD(\s*RES)?', r'ABDO', r'ABDOMEN'],
    "SaO2":     [r'(SaO2|SpO2|OXSTAT|OXIMETRY?)']
}

random.seed(SEED); np.random.seed(SEED)

# =========================
# UTILS
# =========================
def check_mem():
    mem = psutil.virtual_memory()
    pct = mem.percent; avail = mem.available/1024**3
    print(f"💾 Mem: {pct:.1f}% used | {avail:.1f} GB free")
    if pct > MAX_MEMORY_PCT:
        gc.collect()
        return False
    return True

def list_subjects(edf_dir: Path, xml_dir: Path, prefix: str):
    """Return sorted subject IDs that have both EDF and XML present.
       Expects filenames like 'shhs1-XXXX.edf' / 'shhs1-XXXX-nsrr.xml' or 'shhs2-XXXX.*'."""
    ids = []
    for edf in edf_dir.glob(f"{prefix}-*.edf"):
        sid = edf.stem.replace(f"{prefix}-", "")
        xml = xml_dir / f"{prefix}-{sid}-nsrr.xml"
        if xml.exists():
            ids.append(sid)
    ids = sorted(set(ids))
    return ids

def ahi_from_xml(xml_path: Path):
    """Compute TST (s), event count, and AHI from XML."""
    tst = 0.0; n_events = 0
    try:
        root = ET.parse(xml_path).getroot()
        for ev in root.findall(".//ScoredEvent"):
            concept = (ev.findtext("EventConcept") or "").lower().strip()
            if concept in {"stage n1","stage n2","stage n3","stage rem"}:
                dur = float(ev.findtext("Duration") or 30.0)
                tst += dur
            lbl = norm_event(ev.findtext("EventConcept"))
            if lbl is not None:
                dur = float(ev.findtext("Duration") or 0.0)
                if dur >= 10.0: n_events += 1
        ahi = n_events / max(tst/3600.0, 1e-6) if tst>0 else np.nan
        return tst, n_events, ahi
    except Exception:
        return 0.0, 0, np.nan

def severity_from_ahi(ahi):
    if not np.isfinite(ahi): return 0
    if ahi < 5: return 0
    if ahi < 15: return 1
    if ahi < 30: return 2
    return 3

def make_shhs1_train_val(shhs1_ids, xml_dir, val_ratio=0.2, seed=42):
    """Stratified by severity (computed from XML)."""
    bins = {0:[], 1:[], 2:[], 3:[]}
    for sid in shhs1_ids:
        xml = xml_dir / f"shhs1-{sid}-nsrr.xml"
        _, _, ahi = ahi_from_xml(xml)
        sev = severity_from_ahi(ahi)
        bins[sev].append(sid)
    train_ids, val_ids = [], []
    rng = np.random.default_rng(seed)
    for sev, arr in bins.items():
        arr = list(sorted(arr))
        rng.shuffle(arr)
        k = int(round(len(arr)*val_ratio))
        val_ids += arr[:k]
        train_ids += arr[k:]
    return sorted(train_ids), sorted(val_ids)

def resolve_channels(raw):
    mapping = {}
    for want, pats in CANDIDATES.items():
        for ch in raw.ch_names:
            for p in pats:
                if re.fullmatch(p, ch, re.I):
                    mapping[want] = ch; break
            if want in mapping: break
    # require airflow; belts preferred; SaO2 optional but recommended
    required = ["NEW AIR"]
    if not all(k in mapping for k in required):
        return None
    keep = [mapping["NEW AIR"]]
    for k in ["THOR RES","ABDO RES","SaO2"]:
        if k in mapping: keep.append(mapping[k])
    return keep

def normalize_channels(raw):
    """Resample & normalize: z-score airflow/belts; SaO2 clipped to [50,100]/100."""
    raw.resample(TARGET_FS)
    data = raw.get_data()
    for i, ch in enumerate(raw.ch_names):
        if re.search(r'(SaO2|SpO2|OX)', ch, re.I):
            data[i] = np.clip(data[i], 50, 100) / 100.0
        else:
            mu, sd = data[i].mean(), data[i].std() + 1e-8
            data[i] = (data[i] - mu) / sd
    raw._data = data
    return raw

# =========================
# XML PARSERS
# =========================
def norm_event(lbl: str):
    s = (lbl or "").lower()
    if "obstructive" in s and "apnea" in s: return "obstructive_apnea"
    if "central" in s and "apnea" in s:     return "central_apnea"
    if "hypopnea" in s:                     return "hypopnea"
    return None

def parse_events(xml_path):
    """Return sorted list of (start, end, cls_str) for ObA/CnA/Hyp (dur>=10s)."""
    evts = []
    try:
        root = ET.parse(xml_path).getroot()
        for ev in root.findall(".//ScoredEvent"):
            lbl = norm_event(ev.findtext("EventConcept"))
            if lbl is None: continue
            start = float(ev.findtext("Start") or 0.0)
            dur   = float(ev.findtext("Duration") or 0.0)
            if dur >= 10.0:
                evts.append((start, start+dur, lbl))
    except Exception as e:
        print(f"  ❌ XML error {xml_path.name}: {e}")
    evts.sort(key=lambda x: x[0])
    return evts

def parse_sleep_time(xml_path, epoch_len=30.0):
    """Compute total sleep time from stage events (N1/N2/N3/REM)."""
    sleep = 0.0
    try:
        root = ET.parse(xml_path).getroot()
        for ev in root.findall(".//ScoredEvent"):
            concept = (ev.findtext("EventConcept") or "").lower().strip()
            if concept in {"stage n1","stage n2","stage n3","stage rem"}:
                dur = float(ev.findtext("Duration") or epoch_len)
                sleep += dur
    except Exception as e:
        print(f"  ⚠️ Staging parse warn {xml_path.name}: {e}")
    return sleep  # seconds

# =========================
# WINDOW LABELING
# =========================
def label_by_center(events, t_center):
    for s,e,cls in events:
        if s <= t_center < e:
            return LBL_MAP[cls]
        if s > t_center:
            break
    return LBL_MAP["none"]

def extract_window(raw, t0, dur):
    fs = raw.info['sfreq']
    s0 = int(max(0, t0*fs))
    s1 = int(min(raw.n_times, s0 + int(dur*fs)))
    if s1 - s0 < int(dur*fs): return None
    data, _ = raw[:, s0:s1]
    return data.astype(np.float32)

# =========================
# SUBJECT PROCESSOR
# =========================
def process_subject(prefix, sid, split, edf_dir, xml_dir):
    edf = edf_dir / f"{prefix}-{sid}.edf"
    xml = xml_dir / f"{prefix}-{sid}-nsrr.xml"
    if not edf.exists() or not xml.exists():
        return 0

    try:
        raw = read_raw_edf(edf, preload=True, verbose=False)
        keep = resolve_channels(raw)
        if not keep:
            print(f"  ⚠️ Missing required channels in {prefix}-{sid}")
            return 0
        raw.pick_channels(keep)
        raw = normalize_channels(raw)
    except Exception as e:
        print(f"  ❌ EDF error {prefix}-{sid}: {e}")
        return 0

    events = parse_events(xml)
    if events is None:
        return 0

    # Whole-night sliding windows
    fs = raw.info['sfreq']
    T  = raw.n_times / fs
    win = WINDOW_SEC
    st  = STRIDE_SEC

    Xs, ys, metas = [], [], []
    t = 0.0
    while t + win <= T:
        center = t + win/2.0
        y = label_by_center(events, center)
        W = extract_window(raw, t, win)
        if W is not None:
            Xs.append(W)         # [C, win*fs]
            ys.append(y)
            metas.append((float(t), float(center)))
        t += st
        if len(Xs) % 512 == 0 and not check_mem():
            break

    # Save per subject as one shard (fewer files)
    out_dir = OUT_DIR / split
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{prefix}-{sid}_win{int(WINDOW_SEC)}s_stride{int(STRIDE_SEC)}s_fs{int(TARGET_FS)}.npz"
    if Xs:
        X = np.stack(Xs)              # [N, C, T]
        y = np.array(ys, dtype=np.int16)
        meta = {
            "dataset": prefix,
            "subject": sid,
            "channels": raw.ch_names,
            "fs": fs,
            "window_sec": WINDOW_SEC,
            "stride_sec": STRIDE_SEC,
            "n_windows": int(len(Xs)),
            "meta_times": metas,      # (t0, center)
        }
        np.savez_compressed(out_path, X=X, y=y, meta=json.dumps(meta))
        n_saved = len(Xs)
    else:
        n_saved = 0

    # per-night targets for eval
    tst_sec, n_true_events, ahi_true = ahi_from_xml(xml)
    with open(OUT_DIR / "night_targets.csv", "a", encoding="utf-8") as f:
        f.write(f"{prefix},{sid},{split},{tst_sec:.1f},{n_true_events},{ahi_true:.3f}\n")

    del raw, Xs, ys
    gc.collect()
    return n_saved

# =========================
# MAIN
# =========================
def main():
    # 1) Discover subjects
    shhs1_ids = list_subjects(SHHS1_EDF, SHHS1_XML, "shhs1")
    shhs2_ids = list_subjects(SHHS2_EDF, SHHS2_XML, "shhs2")

    if not shhs1_ids:
        print("❌ No SHHS1 subjects found."); return
    if not shhs2_ids:
        print("❌ No SHHS2 subjects found."); return

    # 2) Remove overlaps (strict independence)
    #    SHHS2 IDs are separate namespace, but their numeric tails overlap participants.
    #    We exclude any SHHS1 subject whose numeric ID exists in SHHS2.
    shhs2_numeric = set(shhs2_ids)
    shhs1_clean = [sid for sid in shhs1_ids if sid not in shhs2_numeric]

    print(f"👥 SHHS1 total: {len(shhs1_ids)} | after excluding SHHS2 overlaps: {len(shhs1_clean)}")
    print(f"🧪 SHHS2 test set size: {len(shhs2_ids)}")

    # 3) Stratified SHHS1 train/val by severity
    train_ids, val_ids = make_shhs1_train_val(shhs1_clean, SHHS1_XML, val_ratio=VAL_RATIO, seed=SEED)
    print(f"📚 Train: {len(train_ids)} | 🔎 Val: {len(val_ids)} | 🧪 Test (SHHS2): {len(shhs2_ids)}")

    # 4) Build datasets
    print("\n🚀 Building 60s/30s sliding-window dataset (4-class, label by center)")
    totals = {"train":0, "val":0, "test":0}
    for split, ids, (edf_dir, xml_dir, prefix) in [
        ("train", train_ids, (SHHS1_EDF, SHHS1_XML, "shhs1")),
        ("val",   val_ids,   (SHHS1_EDF, SHHS1_XML, "shhs1")),
        ("test",  shhs2_ids, (SHHS2_EDF, SHHS2_XML, "shhs2")),
    ]:
        pbar = tqdm(ids, desc=f"{split.upper()} subjects", unit="subj")
        for sid in pbar:
            n = process_subject(prefix, sid, split, edf_dir, xml_dir)
            totals[split] += n
            pbar.set_postfix_str(f"wins+={n}, total={totals[split]}")
            if not check_mem():
                print("🛑 Stopping due to memory threshold.")
                break

    print("\n🎉 Done.")
    print(f"📦 Windows saved — Train: {totals['train']} | Val: {totals['val']} | Test: {totals['test']}")
    print(f"📄 Night targets at: {OUT_DIR/'night_targets.csv'}")
    print(f"🗂 Output dirs: {OUT_DIR/'train'}, {OUT_DIR/'val'}, {OUT_DIR/'test'}")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n🛑 Interrupted by user")


👥 SHHS1 total: 5793 | after excluding SHHS2 overlaps: 3146
🧪 SHHS2 test set size: 2651
📚 Train: 2517 | 🔎 Val: 629 | 🧪 Test (SHHS2): 2651

🚀 Building 60s/30s sliding-window dataset (4-class, label by center)


TRAIN subjects:   0%|          | 0/2517 [00:00<?, ?subj/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 52.9% used | 15.0 GB free
💾 Mem: 53.0% used | 15.0 GB free
💾 Mem: 52.9% used | 15.0 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 53.2% used | 14.9 GB free
💾 Mem: 53.2% used | 14.9 GB free
💾 Mem: 52.7% used | 15.1 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 52.9% used | 15.0 GB free
💾 Mem: 52.9% used | 15.0 GB free
💾 Mem: 52.8% used | 15.1 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 52.3% used | 15.2 GB free
💾 Mem: 52.3% used | 15.2 GB free
💾 Mem: 52.1% used | 15.3 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 52.3% used | 15.2 GB free
💾 Mem: 52.2% used | 15.3 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 52.1% used | 15.3 GB free
💾 Mem: 51.9% used | 15.4 

VAL subjects:   0%|          | 0/629 [00:00<?, ?subj/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 51.2% used | 15.6 GB free
💾 Mem: 51.2% used | 15.6 GB free
💾 Mem: 51.1% used | 15.6 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 51.0% used | 15.6 GB free
💾 Mem: 50.8% used | 15.7 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 50.9% used | 15.7 GB free
💾 Mem: 51.0% used | 15.6 GB free
💾 Mem: 50.9% used | 15.7 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 50.9% used | 15.7 GB free
💾 Mem: 50.9% used | 15.7 GB free
💾 Mem: 50.9% used | 15.7 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 51.1% used | 15.6 GB free
💾 Mem: 51.1% used | 15.6 GB free
💾 Mem: 51.0% used | 15.6 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 51.1% used | 15.6 GB free
💾 Mem: 51.1% used | 15.6 

TEST subjects:   0%|          | 0/2651 [00:00<?, ?subj/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.6% used | 14.5 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 54.9% used | 14.4 GB free
💾 Mem: 55.0% used | 14.3 GB free
💾 Mem: 55.0% used | 14.4 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.8% used | 14.4 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 55.2% used | 14.3 GB free
💾 Mem: 55.2% used | 14.3 GB free
💾 Mem: 55.0% used | 14.3 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.8% used | 14.4 GB free
💾 Mem: 54.5% used | 14.5 GB free
NOTE: pick_channels() is a legacy function. New code should use inst.pic

# SHHS1 Event-Level Dataset Builder — Summary

## 🕒 Windowing
- **Window length:** 60 s  
- **Stride:** 30 s (50% overlap)  
- **Coverage:** Full-night continuous sliding windows

## 🏷️ Labeling Strategy
- **Label rule:** Window label = **event covering the window center time**
- **Classes:** {ObA, CnA, Hyp, Non-event}
- **Class map:**  
  - `0 = ObA (Obstructive Apnea)`  
  - `1 = CnA (Central Apnea)`  
  - `2 = Hyp (Hypopnea)`  
  - `3 = None (Non-event)`

## 📡 Channels (SHHS1)
- `NEW AIR` — Nasal airflow / pressure  
- `THOR RES` — Thoracic effort belt  
- `ABDO RES` — Abdominal effort belt  
- `SaO₂` — Pulse oximetry  
- **Robust resolver** supports SHHS1 name variants (e.g., AIRFLOW, NASAL PRES, THORAX, ABDOMEN, SpO₂/OXSTAT).

## ⚙️ Signal Preprocessing
- **Resample:** 100 Hz (uniform across channels)  
- **Normalize:**  
  - Airflow & belts → z-score per channel  
  - SaO₂ → clip to [50, 100], then scale to 0–1

## 💾 Output Structure
- **Per-subject shard (.npz):**
  - `X` → `[N, C, T]` signal windows  
  - `y` → `[N]` integer labels (0–3)  
  - `meta` → JSON (fs, channel names, window start & center times, counts)
- **Global CSV:** `night_targets.csv` with total sleep time (TST), true event count, and true AHI.

## ✅ Design Highlights
- 60 s / 30 s sliding windows with **label-by-center** for clean handling of isolated/adjacent events  
- Minimal 4-channel set aligned with AASM respiratory definitions  
- Memory-safe batching; compressed per-subject shards  
- Train **one 4-class event detector** → derive **AHI** and **severity** from detections
